# Answer-Structure Fine-Tuning — Multi-Arm Experiment (Llama 3.2 3B, LoRA)

**Purpose.** The main study found that giving an *untuned* small model an explicit answer-structure schema did **not** improve extractive QA and usually hurt it, while the source passage helped. The concluded reason was **delivery**: an explicit schema competes with the implicit structure a model already reads in prose (§6.2/§6.3). This notebook tests, across several training arms, whether a model *taught* to read the structure can use it — and whether structure can approach a paragraph's effectiveness without a passage (§6.6).

**Method.** LoRA on `meta-llama/Llama-3.2-3B` (base). LoRA freezes all 3.2B original weights and trains small low-rank adapters on the attention/MLP projections (<1% of params). It is reversible: `disable_adapter()` recovers the untuned base model, which we use for an in-framework baseline.

**Arms (one LoRA adapter each):**
- **Arm 1 — empty-schema format:** train on a mix of `{question}` and `{question, empty matched schema}` → answer. Does the bare matched schema help once the QA format is learned? (Trained analog of the main study.)
- **Arm 2 — schema-sensitivity:** train with matched *and* mismatched empty schemas (same target). Eval C3 vs C5 tests whether the model learns to prefer the correct structure — the merit test.
- **Arm 4 — answer-only control:** train `{question}` → answer, no schema. The floor.
- **Filled-prose delivery** (untuned + Arm S): the filled schema re-rendered as sentences, so the 2x2 (form: prose/schema, content: empty/filled) is complete on the untuned model. No new training arm; deterministic template.
- **Content retention** measure on the filled schemas: fraction of gold answers recoverable from the filled reasoning fields (Gap C). Bounds the content-loss confound when reading Arm S vs the passage.
- **Bootstrap 95% CIs** reported on the key pairwise deltas so each claim is testable, not a bare number.

- **Arm S — filled schema in the input:** train `{question, FILLED schema}` → answer, where the schema dimensions carry content drawn from the passage (by Claude). Eval head-to-head vs the untuned passage condition (C2 ~0.906). Tests whether the answer structure, when it *carries* the content, is a form the model can use as well as prose. **Caveat:** the filled schema is a lossy re-encoding of the passage, so content is not held perfectly equal; read Arm S as evidence about structure-vs-content, not proof.

**Eval (all arms):** the fixed 1,000-question sample, no passage, at C1 (question), C3 (matched empty schema), C5 (mismatched empty schema), scored EM_ext / F1_ext (identical to the main study). Plus a **C2 passage probe** on the Arm 1 model (a condition it was never trained on) to check whether schema training narrowed its passage ability.

**Reference (untuned 3B, main study, Ollama):** C1 F1_ext 0.369 | C2 (passage) 0.906 | C3 0.343 | C5 0.360. The C2 value is the **paragraph target** structure must try to approach.

**Files to place in `MyDrive/answer_structure_finetune_in/`:** `arm1_train.jsonl`, `arm1_val.jsonl`, `arm2_train.jsonl`, `arm2_val.jsonl`, `arm4_train.jsonl`, `arm4_val.jsonl`, `sample_1000.json`, and for Arm S `filled_train_pairs.jsonl`, `filled_eval_pairs.jsonl` (from `taxonomy_classification/populate_schemas.py`).

> Arm S requires the Claude-populated files. If you have not run `populate_schemas.py` yet, skip section 9b; the other arms run without it.

> Use a premium GPU runtime (Runtime → Change runtime type → A100/L4, High-RAM).
> **Staying alive:** keep the browser tab open and your Mac awake (`caffeinate -dimsu` in a local terminal prevents sleep). The real safety net is Drive checkpointing below — `results` and each adapter are saved to Drive after every arm, so if the runtime drops you rerun from section 1 and completed arms reload automatically.

## 1. Environment

In [26]:
# Colab already ships recent torch + transformers with prebuilt wheels.
# Pinning old transformers forces an old `tokenizers` that has no wheel and fails to
# build from source. So: keep Colab's torch/transformers, only add the PEFT stack,
# and use lower-bound pins (>=) so pip picks a wheel that matches the runtime.
!pip -q install -U "peft>=0.12" "datasets>=2.20" "accelerate>=0.33"
# Colab's torchao (0.10) is older than what peft's LoRA dispatcher demands (>=0.16),
# which raises ImportError even though we do not use torchao. We do not quantize,
# so removing torchao makes peft skip that code path cleanly.
!pip -q uninstall -y torchao

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # reduce fragmentation across arms
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

torch 2.11.0+cu128 | transformers 5.16.1 | cuda: True
gpu: NVIDIA A100-SXM4-40GB


## 1b. Runtime check + persistent output (premium Colab)

Confirm the GPU and mount Drive so trained adapters and `results` survive a disconnect. Everything is written to `DRIVE_OUT` after each arm; on reconnect, completed arms reload from the checkpoint instead of retraining.

In [27]:
import torch, subprocess, os, json
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU (A100/L4).'

from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/answer_structure_finetune_out'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('outputs ->', DRIVE_OUT)

def checkpoint_results(results):
    with open(os.path.join(DRIVE_OUT, 'multiarm_results.json'), 'w') as f:
        json.dump(results, f, indent=2)
    print('  [checkpoint] results saved to Drive')

def load_checkpoint():
    p = os.path.join(DRIVE_OUT, 'multiarm_results.json')
    if os.path.exists(p):
        r = json.load(open(p)); print(f'  [resume] loaded {len(r)} prior result keys'); return r
    return {}

NVIDIA A100-SXM4-40GB, 40960 MiB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
outputs -> /content/drive/MyDrive/answer_structure_finetune_out


In [5]:
from huggingface_hub import login
login()  # HF token with access to meta-llama/Llama-3.2-3B (accept the license first)

## 2. Data from Drive

Put the data files in a Drive folder `MyDrive/answer_structure_finetune_in/` **once** (drag-and-drop in the Drive web UI, or copy from your machine). The run reads them from there, so a runtime disconnect never means re-uploading. Required: the six arm files + `sample_1000.json`. Optional (Arm S): `filled_train_pairs.jsonl`, `filled_eval_pairs.jsonl`.

In [28]:
# Read ALL inputs from a Drive folder (not files.upload(), which is ephemeral and
# dies with the runtime). Put the data files in DRIVE_IN once; the run then survives
# disconnects for both inputs (here) and outputs (DRIVE_OUT). Drive is mounted in 1b.
DRIVE_IN = '/content/drive/MyDrive/answer_structure_finetune_in'
DATA = DRIVE_IN

def read_jsonl(p):
    with open(p) as f:
        return [json.loads(l) for l in f if l.strip()]

# Preflight: report which required / optional files are present.
required = ['arm1_train.jsonl','arm1_val.jsonl','arm2_train.jsonl','arm2_val.jsonl',
            'arm4_train.jsonl','arm4_val.jsonl','sample_1000.json']
optional_armS = ['filled_train_pairs.jsonl','filled_eval_pairs.jsonl']
print('DRIVE_IN =', DRIVE_IN)
missing = [f for f in required if not os.path.exists(os.path.join(DATA, f))]
for f in required:
    print(('  [ok] ' if f not in missing else '  [MISSING] ') + f)
for f in optional_armS:
    print(('  [ok] ' if os.path.exists(os.path.join(DATA, f)) else '  [absent, Arm S will skip] ') + f)
assert not missing, f'Put these in {DRIVE_IN}: {missing}'

SAMPLE_PATH = os.path.join(DATA, 'sample_1000.json')
sample = json.load(open(SAMPLE_PATH))['questions']
print('eval sample:', len(sample), '| example keys:', list(sample[0].keys()))
assert 'passage' in sample[0] and 'class_id' in sample[0], 'sample must carry passage + class_id'

DRIVE_IN = /content/drive/MyDrive/answer_structure_finetune_in
  [ok] arm1_train.jsonl
  [ok] arm1_val.jsonl
  [ok] arm2_train.jsonl
  [ok] arm2_val.jsonl
  [ok] arm4_train.jsonl
  [ok] arm4_val.jsonl
  [ok] sample_1000.json
  [ok] filled_train_pairs.jsonl
  [ok] filled_eval_pairs.jsonl
eval sample: 1000 | example keys: ['id', 'question', 'passage', 'gold_answers', 'class_id', 'features']


## 3. Embedded scaffold builder, schemas, and scoring

Copied from the repo (`scaffold_builder.py`, `schemas.json`, `run_condition.py`), verified byte-identical, so the notebook is self-contained and metrics match the main study.

In [29]:
import random, re, string

SCHEMAS = {
  1:  {"answer_type": "single",   "answer": "object",            "object": "", "relation": ""},
  2:  {"answer_type": "single",   "answer": "concept or person", "identity": "", "defining_property": ""},
  3:  {"answer_type": "multiple", "answer": "list",              "components": [], "shared_category": ""},
  4:  {"answer_type": "single",   "answer": "quantity",          "quantity": "", "unit": "", "measurement_context": ""},
  5:  {"answer_type": "single",   "answer": "time",              "event": "", "time_expression": "", "temporal_type": ""},
  6:  {"answer_type": "single",   "answer": "location",          "location": "", "relation": ""},
  7:  {"answer_type": "single",   "answer": "agent",             "agent": "", "action_performed": "", "object_of_action": ""},
  8:  {"answer_type": "single",   "answer": "quality",           "quality": "", "description_domain": ""},
  9:  {"answer_type": "single",   "answer": "cause",             "effect": "", "cause": "", "mechanism": ""},
  10: {"answer_type": "single",   "answer": "outcome",           "action": "", "actor": "", "outcome": ""},
}

def get_answer_structure(cid): return SCHEMAS[cid]
def get_random_class_id(exclude_id): return random.choice([c for c in SCHEMAS if c != exclude_id])

def build_scaffold(question, condition, passage=None, class_id=None, random_seed=None):
    needs_passage = condition in (2, 4, 6, 8)
    needs_structure = condition in (3, 4, 5, 6, 7, 8)
    scaffold = {"question": question}
    if needs_structure:
        if condition in (5, 6):
            if random_seed is not None:
                random.seed(random_seed)
            scaffold["answer_structure"] = get_answer_structure(get_random_class_id(exclude_id=class_id))
        else:
            scaffold["answer_structure"] = get_answer_structure(class_id)
    if needs_passage:
        scaffold["context_passage"] = passage
    return json.dumps(scaffold, indent=2)

def normalize_answer(text):
    text = text.lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())

def _token_f1(p, g):
    if not p or not g: return 0.0
    common = set(p) & set(g)
    nc = sum(min(p.count(t), g.count(t)) for t in common)
    if nc == 0: return 0.0
    prec, rec = nc/len(p), nc/len(g)
    return 2*prec*rec/(prec+rec)

def best_span_extraction(prediction, gold_answers):
    pt = normalize_answer(prediction).split()
    if not pt: return ""
    gtl = [normalize_answer(g).split() for g in gold_answers]
    mgl = max((len(g) for g in gtl), default=1)
    msl = min(len(pt), mgl*2)
    best_f1, best = 0.0, pt
    for s in range(len(pt)):
        for e in range(s+1, min(s+msl+1, len(pt)+1)):
            span = pt[s:e]
            for gt in gtl:
                f1 = _token_f1(span, gt)
                if f1 > best_f1: best_f1, best = f1, span
    return " ".join(best)

def exact_match(p, g): return 1.0 if any(normalize_answer(p)==normalize_answer(x) for x in g) else 0.0
def exact_match_extracted(p, g):
    ex = best_span_extraction(p, g); return 1.0 if any(ex==normalize_answer(x) for x in g) else 0.0
def f1_score(p, g):
    pt = normalize_answer(p).split(); return max((_token_f1(pt, normalize_answer(x).split()) for x in g), default=0.0)
def f1_extracted(p, g):
    ex = best_span_extraction(p, g).split(); return max((_token_f1(ex, normalize_answer(x).split()) for x in g), default=0.0)

# --- alternative-prose delivery: render the EMPTY schema as sentences (deterministic, no LLM) ---
# Class name + description + the dimension field names, phrased as guidance. This is the
# "structure as sentences" form: same information as the empty schema, prose instead of JSON.
CLASS_META = {
  1:  ("Object Identification", "The answer names or identifies a concrete object, entity, document, technology, or substance."),
  2:  ("Concept/Person Identification", "The answer identifies an abstract concept, classification, or person by a defining property."),
  3:  ("Multipart Identification", "The answer requires identifying multiple distinct components."),
  4:  ("Quantitative", "The answer is a number: a count, measurement, proportion, or recorded statistic."),
  5:  ("Temporal", "The answer is a time expression: a date, year, era, duration, or range."),
  6:  ("Place", "The answer is a geographic location or spatial reference."),
  7:  ("Agent", "The answer is who or what performed an action."),
  8:  ("Descriptive", "The answer describes a quality, property, or characteristic of something."),
  9:  ("Causal", "The answer explains why something happened: the cause, reason, or enabling mechanism."),
  10: ("Resultative", "The answer is the outcome, effect, or product of an action or event."),
}

def _dimension_names(class_id):
    # the empty (non-prefilled) dimensions of the schema, in order
    schema = SCHEMAS[class_id]
    return [k for k in schema if k not in ("answer_type", "answer")]

def render_prose(question, class_id):
    """Alternative-prose scaffold: same info as the empty schema, in sentence form."""
    name, desc = CLASS_META[class_id]
    dims = _dimension_names(class_id)
    if len(dims) == 1:
        dim_sentence = f"A complete answer addresses the {dims[0].replace('_',' ')}."
    elif len(dims) == 2:
        dim_sentence = f"A complete answer addresses the {dims[0].replace('_',' ')} and {dims[1].replace('_',' ')}."
    else:
        joined = ", ".join(d.replace("_", " ") for d in dims[:-1]) + f", and {dims[-1].replace('_',' ')}"
        dim_sentence = f"A complete answer addresses the {joined}."
    return (f"Question: {question}\n"
            f"This is a {name} question. {desc} {dim_sentence}")

# --- Gap A: render a FILLED schema as sentences (deterministic, no LLM) ---
def _fmt_value(v):
    if isinstance(v, list):
        return ", ".join(str(x) for x in v if str(x).strip()) or "(none)"
    return str(v) if str(v).strip() else "(none)"

def render_filled_prose(question, filled_schema):
    """Filled-prose scaffold: same content as the filled JSON schema, in sentence form."""
    cid = None
    ans_type_hint = filled_schema.get("answer") if isinstance(filled_schema, dict) else None
    for k, meta in CLASS_META.items():
        if SCHEMAS[k]["answer"] == ans_type_hint:
            cid = k; break
    if cid is None: cid = 1
    name, desc = CLASS_META[cid]
    dims = _dimension_names(cid)
    parts = [f"Question: {question}", f"This is a {name} question. {desc}"]
    for d in dims:
        parts.append(f"The {d.replace('_',' ')} is: {_fmt_value(filled_schema.get(d, ''))}.")
    # answer_value (if present) is deliberately NOT surfaced in the prose — it would give the answer away.
    return "\n".join(parts)

# --- Gap C: content-retention on filled schemas ---
def gold_recoverable_from_filled(filled_schema, gold_answers):
    """Cheap proxy: does any gold answer (normalized) appear as a substring of the
    concatenated filled-field values (excluding answer_value which trivially contains it)?"""
    if not isinstance(filled_schema, dict): return False
    parts = []
    for k, v in filled_schema.items():
        if k in ("answer_type", "answer", "answer_value"): continue
        if isinstance(v, list):
            parts.extend(str(x) for x in v)
        else:
            parts.append(str(v))
    blob = normalize_answer(" ".join(parts))
    return any(normalize_answer(g) and normalize_answer(g) in blob for g in gold_answers)

# --- Tightness: bootstrap 95% CI on the mean of a per-question metric list ---
def bootstrap_ci(values, n_boot=1000, alpha=0.05, seed=0):
    if not values: return (None, None, None)
    import random as _r
    rng = _r.Random(seed)
    m = sum(values)/len(values)
    boots = []
    N = len(values)
    for _ in range(n_boot):
        s = 0.0
        for _ in range(N):
            s += values[rng.randrange(N)]
        boots.append(s / N)
    boots.sort()
    lo = boots[int((alpha/2)*n_boot)]
    hi = boots[int((1-alpha/2)*n_boot)]
    return (round(m,4), round(lo,4), round(hi,4))

def bootstrap_delta_ci(a_vals, b_vals, n_boot=1000, alpha=0.05, seed=0):
    """Paired bootstrap CI on mean(a) - mean(b) with matched samples (same q order)."""
    import random as _r
    rng = _r.Random(seed)
    N = min(len(a_vals), len(b_vals))
    diffs = [a_vals[i]-b_vals[i] for i in range(N)]
    m = sum(diffs)/N
    boots = []
    for _ in range(n_boot):
        s = 0.0
        for _ in range(N):
            s += diffs[rng.randrange(N)]
        boots.append(s/N)
    boots.sort()
    lo = boots[int((alpha/2)*n_boot)]
    hi = boots[int((1-alpha/2)*n_boot)]
    return (round(m,4), round(lo,4), round(hi,4))

# --- Arm S leakage fix: answer_value is the GOLD answer; it must NOT be in the model input.
# Strip it from the scaffold's answer_structure (keep the passage-derived reasoning fields).
def strip_answer_value(scaffold_json):
    d = json.loads(scaffold_json)
    if isinstance(d.get('answer_structure'), dict):
        d['answer_structure'] = {k: v for k, v in d['answer_structure'].items() if k != 'answer_value'}
    return json.dumps(d, indent=2)


## 4. Tokenizer, prompt format, and dataset builder

Prompt = the scaffold JSON + `Answer:` tag; loss masked on the prompt so the model only learns to produce the answer. The `input` field in each arm file already contains the scaffold, so tokenization is arm-agnostic.

In [30]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_ID = 'meta-llama/Llama-3.2-3B'
ANSWER_TAG = '\n\nAnswer: '
MAX_LEN = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def build_prompt(scaffold_input):
    return scaffold_input + ANSWER_TAG

def tokenize_example(row):
    prompt = build_prompt(row['input'])
    target = row['target'] + tokenizer.eos_token
    pids = tokenizer(prompt, add_special_tokens=True)['input_ids']
    tids = tokenizer(target, add_special_tokens=False)['input_ids']
    ids = (pids + tids)[:MAX_LEN]
    labels = ([-100]*len(pids) + tids)[:MAX_LEN]
    return {'input_ids': ids, 'attention_mask': [1]*len(ids), 'labels': labels}

def load_arm_dataset(train_file, val_file):
    tr = read_jsonl(os.path.join(DATA, train_file))
    va = read_jsonl(os.path.join(DATA, val_file))
    return (Dataset.from_list([tokenize_example(r) for r in tr]),
            Dataset.from_list([tokenize_example(r) for r in va]),
            len(tr), len(va))

## 5. Reusable train + eval functions

`train_arm` loads a **fresh** base model, wraps it in a new LoRA adapter, trains, and returns the PEFT model. `evaluate` runs greedy decoding over the sample for a given condition. Because each arm reloads the base model, adapters do not contaminate each other.

In [31]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model
import gc

LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
            target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
TRAIN_ARGS = dict(num_train_epochs=3, per_device_train_batch_size=8, gradient_accumulation_steps=2,
                  learning_rate=2e-4, logging_steps=25, save_strategy='no', bf16=True, report_to='none')
OPTIONAL_TRAIN_ARGS = dict(lr_scheduler_type='cosine', warmup_ratio=0.03)  # added only if supported

import inspect
def make_training_args(output_dir):
    """Build TrainingArguments passing only kwargs this transformers version accepts.
    Handles the eval_strategy/evaluation_strategy rename and any dropped args."""
    sig = set(inspect.signature(TrainingArguments.__init__).parameters)
    args = dict(TRAIN_ARGS)
    for k, v in OPTIONAL_TRAIN_ARGS.items():
        if k in sig: args[k] = v
    # handle the eval_strategy <-> evaluation_strategy rename
    if 'eval_strategy' not in sig and 'evaluation_strategy' in sig and 'eval_strategy' in args:
        args['evaluation_strategy'] = args.pop('eval_strategy')
    filtered = {k: v for k, v in args.items() if k in sig}
    dropped = [k for k in args if k not in sig]
    if dropped:
        print('  (dropping unsupported TrainingArguments:', dropped, ')')
    return TrainingArguments(output_dir=output_dir, **filtered)

GEN_BATCH = 32          # questions per generate() call during eval
MAX_NEW_TOKENS = 32     # SQuAD answers are short; span extraction recovers the rest

def fresh_base_model():
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
    m.config.pad_token_id = tokenizer.pad_token_id
    return m

def train_arm(name, train_file, val_file, batch_size=None, grad_accum=None):
    print(f'\n===== TRAIN {name} =====')
    tokenizer.padding_side = 'right'  # correct for training/label alignment
    train_ds, val_ds, ntr, nva = load_arm_dataset(train_file, val_file)
    print(f'  train={ntr} val={nva}')
    model = get_peft_model(fresh_base_model(), LoraConfig(**LORA))
    model.print_trainable_parameters()
    collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100, padding=True)
    ta = make_training_args(f'out_{name}')
    if batch_size is not None: ta.per_device_train_batch_size = batch_size
    if grad_accum is not None: ta.gradient_accumulation_steps = grad_accum
    trainer = Trainer(model=model, args=ta,
                      train_dataset=train_ds, eval_dataset=val_ds, data_collator=collator)
    trainer.train()
    model.save_pretrained(f'adapter_{name}')
    try:
        import shutil
        if 'DRIVE_OUT' in globals():
            dst = os.path.join(DRIVE_OUT, f'adapter_{name}')
            if os.path.exists(dst): shutil.rmtree(dst)
            shutil.copytree(f'adapter_{name}', dst)
            print(f'  [checkpoint] adapter_{name} -> Drive')
    except Exception as e:
        print('  (adapter Drive-copy skipped:', e, ')')
    return model

@torch.no_grad()
def generate_batch(m, prompts, max_new_tokens=MAX_NEW_TOKENS, batch_size=GEN_BATCH):
    """Batched greedy generation. Uses LEFT padding (required so the generated tokens
    align to the right edge for a decoder), then restores the training padding side.
    Returns the first non-empty line of each completion."""
    prev_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    outputs = []
    try:
        for s in range(0, len(prompts), batch_size):
            chunk = [build_prompt(p) for p in prompts[s:s+batch_size]]
            enc = tokenizer(chunk, return_tensors='pt', padding=True,
                            truncation=True, max_length=MAX_LEN).to(m.device)
            gen = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
                             pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
            new = gen[:, enc['input_ids'].shape[1]:]           # strip the (left-padded) prompt
            for row in tokenizer.batch_decode(new, skip_special_tokens=True):
                outputs.append(row.strip().split('\n')[0].strip())
    finally:
        tokenizer.padding_side = prev_side
    return outputs

@torch.no_grad()
def generate_answer(m, scaffold_input, max_new_tokens=MAX_NEW_TOKENS):
    """Single-item convenience wrapper (kept for ad-hoc use); eval uses generate_batch."""
    return generate_batch(m, [scaffold_input], max_new_tokens=max_new_tokens, batch_size=1)[0]

def _score(preds, golds, condition, n):
    agg = dict(em=0.0, em_ext=0.0, f1=0.0, f1_ext=0.0, tok=0.0)
    per_q_f1ext = []
    for pred, g in zip(preds, golds):
        _f1e = f1_extracted(pred, g)
        agg['em'] += exact_match(pred, g); agg['em_ext'] += exact_match_extracted(pred, g)
        agg['f1'] += f1_score(pred, g);    agg['f1_ext'] += _f1e
        agg['tok'] += len(pred.split()); per_q_f1ext.append(_f1e)
    return {'condition': condition, 'n': n, 'per_q_f1ext': per_q_f1ext,
            'em': round(agg['em']/n,4), 'em_ext': round(agg['em_ext']/n,4),
            'f1': round(agg['f1']/n,4), 'f1_ext': round(agg['f1_ext']/n,4),
            'avg_tokens': round(agg['tok']/n,1)}

def evaluate(m, condition, seed=42, limit=None):
    items = sample[:limit] if limit else sample
    scaffolds, golds = [], []
    for i, q in enumerate(items):
        if condition == 'prose':
            sc = render_prose(q['question'], q['class_id'])
        else:
            sc = build_scaffold(q['question'], condition=condition,
                                passage=q['passage'] if condition in (2,4,6,8) else None,
                                class_id=q['class_id'] if condition >= 3 else None,
                                random_seed=seed+i if condition in (5,6) else None)
        scaffolds.append(sc); golds.append(q['gold_answers'])
    print(f'    generating {len(scaffolds)} (batched)...')
    preds = generate_batch(m, scaffolds)
    return _score(preds, golds, condition, len(items))

def free(model):
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def eval_filled(m, filled_eval_file, limit=None, mode='json'):
    """Arm S eval: the scaffold `input` is prebuilt (question + FILLED schema).
    mode='json' feeds the filled JSON; mode='prose' renders it as sentences."""
    gold_by_id = {q['id']: q['gold_answers'] for q in sample}
    rows = read_jsonl(os.path.join(DATA, filled_eval_file))
    rows = [r for r in rows if r['id'] in gold_by_id]
    if limit: rows = rows[:limit]
    scaffolds, golds = [], []
    for r in rows:
        if mode == 'prose':
            filled = json.loads(r['input']).get('answer_structure', {})
            sc = render_filled_prose(r['question'], filled)
        else:
            sc = strip_answer_value(r['input'])  # remove leaked gold answer
        scaffolds.append(sc); golds.append(gold_by_id[r['id']])
    print(f'    generating {len(scaffolds)} (batched, mode={mode})...')
    preds = generate_batch(m, scaffolds)
    return _score(preds, golds, f'S_{mode}', len(rows))

import hashlib
def _passage_hash(t): return hashlib.sha1(t.encode('utf-8')).hexdigest()

def eval_structured_passage(m, restructured_file, limit=None):
    """Meaning 2 / V7 fair test: replace the raw passage with a RESTRUCTURED passage
    (same content, reorganized, answer NOT pre-located). Compare to C2 (raw passage).
    Untuned model only. Scaffold is a passage condition: {question, context_passage=structured}."""
    rp = json.load(open(os.path.join(DATA, restructured_file)))  # {hash: {original, structured}}
    items = sample[:limit] if limit else sample
    scaffolds, golds, missing = [], [], 0
    for q in items:
        h = _passage_hash(q['passage'])
        if h not in rp:
            missing += 1
            structured = q['passage']  # fallback to raw if a passage was not restructured
        else:
            structured = rp[h]['structured']
        # build a C2-style passage scaffold but with the structured passage as the context
        sc = json.dumps({'question': q['question'], 'context_passage': structured}, indent=2)
        scaffolds.append(sc); golds.append(q['gold_answers'])
    if missing: print(f'    (warning: {missing} passages not in restructured file; used raw)')
    print(f'    generating {len(scaffolds)} (batched, structured-passage)...')
    preds = generate_batch(m, scaffolds)
    return _score(preds, golds, 'structured_passage', len(items))

## 6. Untuned HF baseline (removes the Ollama quantization confound)

Evaluate the base model with **no adapter** at C1/C3/C5 and the C2 passage condition, so the fine-tuned arms are compared against an in-framework untuned baseline, not only the Ollama numbers.

In [15]:
results = load_checkpoint()  # resume-aware; empty on first run

if not any(k.startswith('untuned_') for k in results):
    base = fresh_base_model()
    for c in (1, 2, 3, 5):
        print(f'UNTUNED C{c}')
        results[f'untuned_C{c}'] = evaluate(base, c)
        print('  ', results[f'untuned_C{c}'])
    print('UNTUNED prose (alt-prose delivery)')
    results['untuned_prose'] = evaluate(base, 'prose'); print('  ', results['untuned_prose'])
    if os.path.exists(os.path.join(DATA, 'filled_eval_pairs.jsonl')):
        print('UNTUNED filled-schema (base on filled JSON)')
        results['untuned_filled_schema'] = eval_filled(base, 'filled_eval_pairs.jsonl', mode='json')
        print('  ', results['untuned_filled_schema'])
        print('UNTUNED filled-prose (base on filled prose)')
        results['untuned_filled_prose'] = eval_filled(base, 'filled_eval_pairs.jsonl', mode='prose')
        print('  ', results['untuned_filled_prose'])
    else:
        print('(skipping untuned filled-* : filled_eval_pairs.jsonl not present)')
    free(base)
    checkpoint_results(results)
else:
    print('untuned baseline already in results; skipping. Delete untuned_* keys to redo.')

  [resume] loaded 25 prior result keys
untuned baseline already in results; skipping. Delete untuned_* keys to redo.


## 7. Arm 1 — empty-schema format (+ C2 passage probe)

The C2 probe evaluates the Arm 1 model on a passage condition it was never trained on, to check whether schema-format training narrowed its passage ability (§2.3 distribution-narrowing risk).

In [ ]:
if 'arm1_C3' in results:
    print('arm1: already in results, skipping (delete its keys to redo).')
else:
    m1 = train_arm('arm1', 'arm1_train.jsonl', 'arm1_val.jsonl')
    for c in (1, 3, 5):
        print(f'ARM1 C{c}')
        results[f'arm1_C{c}'] = evaluate(m1, c); print('  ', results[f'arm1_C{c}'])
    print('ARM1 C2 (passage probe — off-distribution)')
    results['arm1_C2'] = evaluate(m1, 2); print('  ', results['arm1_C2'])
    print('ARM1 prose (alt-prose delivery)')
    results['arm1_prose'] = evaluate(m1, 'prose'); print('  ', results['arm1_prose'])
    free(m1)
    checkpoint_results(results)


===== TRAIN arm1 =====
  train=8866 val=986


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


Step,Training Loss
25,2.407563
50,2.127792
75,2.009841
100,2.023193
125,1.918011
150,1.935282
175,1.755467
200,1.615664
225,1.767375
250,1.768876


  [checkpoint] adapter_arm1 -> Drive
ARM1 C1
    generating 1000 (batched)...
   {'condition': 1, 'n': 1000, 'per_q_f1ext': [0.4, 0.0, 0.2857142857142857, 1.0, 0.0, 0.0, 0.0, 0.4, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.5, 0.0, 0.0, 0.4, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.6666666666666666, 0.5, 0.0, 0.0, 0.0, 0.2857142857142857, 0.0, 0.0, 0.0, 0.5714285714285715, 0.0, 0.0, 0.5714285714285714, 1.0, 0.0, 0.0, 0.0, 0.0, 0.6666666666666666, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3636363636363636, 0.0, 0.6666666666666666, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.42857142857142855, 0.0, 0.0, 1.0, 0.0, 0.375, 0.4, 0.6666666666666666, 0.0, 0.19999999999999998, 0.0, 0.0, 0.6666666666666666, 0.0, 0.0, 0.4, 0.33333333333333337, 0.0, 0.2857142857142857, 0.6666666666666666, 0.0, 0.23529411764705882, 0.33333333333333337, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.2857142857142857, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.6666666666666666, 0.2857142857142857, 0.0, 0.0, 1.0, 0.0, 0.0, 0.75, 1.0, 0.

## 8. Arm 2 — schema-sensitivity (matched vs mismatched)

In [ ]:
if 'arm2_C3' in results:
    print('arm2: already in results, skipping (delete its keys to redo).')
else:
    m2 = train_arm('arm2', 'arm2_train.jsonl', 'arm2_val.jsonl')
    for c in (1, 3, 5):
        print(f'ARM2 C{c}')
        results[f'arm2_C{c}'] = evaluate(m2, c); print('  ', results[f'arm2_C{c}'])
    print('ARM2 C2 (passage probe)')
    results['arm2_C2'] = evaluate(m2, 2); print('  ', results['arm2_C2'])
    print('ARM2 prose (alt-prose delivery)')
    results['arm2_prose'] = evaluate(m2, 'prose'); print('  ', results['arm2_prose'])
    free(m2)
    checkpoint_results(results)


===== TRAIN arm2 =====
  train=8866 val=986


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


Step,Training Loss
25,2.390885
50,2.136115
75,2.008415
100,2.040726
125,1.944786
150,1.964583
175,1.765689
200,1.598138
225,1.773514
250,1.762096


  [checkpoint] adapter_arm2 -> Drive
ARM2 C1
    generating 1000 (batched)...
   {'condition': 1, 'n': 1000, 'per_q_f1ext': [0.4, 0.0, 0.2857142857142857, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.5, 0.0, 0.25, 0.4, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.6666666666666666, 0.5, 0.0, 0.0, 0.0, 0.2857142857142857, 0.0, 0.2857142857142857, 0.0, 0.5714285714285715, 0.0, 0.0, 0.5714285714285714, 0.8, 0.0, 0.0, 0.0, 1.0, 0.6666666666666666, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3636363636363636, 0.0, 0.6666666666666666, 0.0, 0.0, 0.0, 0.0, 0.4, 0.0, 0.0, 0.8571428571428571, 0.0, 0.0, 1.0, 0.0, 0.4, 0.4, 0.0, 0.0, 0.19999999999999998, 0.0, 0.0, 0.6666666666666666, 0.0, 0.6666666666666666, 0.0, 0.0, 0.14285714285714288, 0.2857142857142857, 0.6666666666666666, 0.0, 0.14285714285714288, 0.5714285714285715, 0.0, 0.0, 1.0, 0.4, 0.0, 0.0, 0.2857142857142857, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6666666666666666, 0.2857142857142857, 0.0, 0.0, 1.0, 0.0, 0.0, 0

## 9. Arm 4 — answer-only control (floor)

In [ ]:
if 'arm4_C3' in results:
    print('arm4: already in results, skipping (delete its keys to redo).')
else:
    m4 = train_arm('arm4', 'arm4_train.jsonl', 'arm4_val.jsonl')
    for c in (1, 3, 5):
        print(f'ARM4 C{c}')
        results[f'arm4_C{c}'] = evaluate(m4, c); print('  ', results[f'arm4_C{c}'])
    print('ARM4 C2 (passage probe)')
    results['arm4_C2'] = evaluate(m4, 2); print('  ', results['arm4_C2'])
    print('ARM4 prose (alt-prose delivery)')
    results['arm4_prose'] = evaluate(m4, 'prose'); print('  ', results['arm4_prose'])
    free(m4)
    checkpoint_results(results)


===== TRAIN arm4 =====
  train=4433 val=493


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


Step,Training Loss
25,2.476697
50,2.045362
75,2.126418
100,2.077659
125,2.026225
150,2.030123
175,2.031172
200,2.050683
225,2.010283
250,1.940547


  [checkpoint] adapter_arm4 -> Drive
ARM4 C1
    generating 1000 (batched)...
   {'condition': 1, 'n': 1000, 'per_q_f1ext': [0.5, 0.0, 0.2857142857142857, 1.0, 0.0, 0.0, 0.0, 0.4, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.5, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.6666666666666666, 0.5, 0.0, 0.0, 0.0, 0.4444444444444444, 0.0, 0.0, 0.0, 0.5714285714285715, 1.0, 0.0, 0.6153846153846153, 1.0, 0.0, 0.4, 0.0, 0.0, 0.6666666666666666, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.25, 0.0, 0.6666666666666666, 0.0, 0.6666666666666666, 1.0, 0.0, 0.4, 0.33333333333333337, 0.0, 0.36363636363636365, 0.0, 0.0, 1.0, 1.0, 0.375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.6666666666666666, 0.0, 0.0, 0.0, 0.0, 0.6666666666666666, 0.0, 0.14285714285714288, 0.0, 0.0, 0.0, 1.0, 0.6666666666666666, 0.0, 0.0, 0.2857142857142857, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2857142857142857, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.2222222222222222,

## 9b. Arm S — filled schema in the input (structured content vs prose)

Train `{question, FILLED schema}` → answer; the filled schema carries passage-derived content. Eval on the filled schemas for the 1,000-question sample and compare against the untuned passage condition (C2). If Arm S approaches C2, structured content is nearly as usable as prose; if it falls well short, form matters beyond content (with the lossy-re-encoding caveat).

Requires `filled_train_pairs.jsonl` and `filled_eval_pairs.jsonl` from `populate_schemas.py` (the val split is carved from the train pool in-cell).

In [22]:
for _k in ('armS_filled','armS_filled_prose','content_retention_filled'):
    results.pop(_k, None)  # force re-run after the leakage fix
if False:
    pass
else:
    import random as _r
    _ft = read_jsonl(os.path.join(DATA, 'filled_train_pairs.jsonl'))
    for _row in _ft:  # strip leaked gold answer from the training input
        _row['input'] = strip_answer_value(_row['input'])
    _r.Random(42).shuffle(_ft)
    _nval = max(1, int(len(_ft) * 0.1))
    _ft_val, _ft_train = _ft[:_nval], _ft[_nval:]
    with open('filled_train_split.jsonl', 'w') as f:
        for r in _ft_train: f.write(json.dumps(r) + '\n')
    with open('filled_val_split.jsonl', 'w') as f:
        for r in _ft_val: f.write(json.dumps(r) + '\n')
    print(f'Arm S train={len(_ft_train)} val={len(_ft_val)}')
    _data_saved = DATA; DATA = '.'
    mS = train_arm('armS', 'filled_train_split.jsonl', 'filled_val_split.jsonl',
                   batch_size=4, grad_accum=4)
    DATA = _data_saved
    print('ARM S (filled schema in input)')
    results['armS_filled']       = eval_filled(mS, 'filled_eval_pairs.jsonl', mode='json')
    print('  ', results['armS_filled'])
    print('ARM S filled-prose (same model, prose rendering of the filled schema)')
    results['armS_filled_prose'] = eval_filled(mS, 'filled_eval_pairs.jsonl', mode='prose')
    print('  ', results['armS_filled_prose'])
    gold_by_id = {q['id']: q['gold_answers'] for q in sample}
    _rows = read_jsonl(os.path.join(DATA, 'filled_eval_pairs.jsonl'))
    _recov = []
    for r in _rows:
        if r['id'] not in gold_by_id: continue
        fs = json.loads(r['input']).get('answer_structure', {})
        _recov.append(1.0 if gold_recoverable_from_filled(fs, gold_by_id[r['id']]) else 0.0)
    if _recov:
        results['content_retention_filled'] = {'n': len(_recov), 'recoverable_fraction': round(sum(_recov)/len(_recov), 4)}
        print('Content retention:', results['content_retention_filled'])
    print('  vs untuned passage C2:', results.get('untuned_C2', {}).get('f1_ext'), '(paragraph target)')
    free(mS)
    checkpoint_results(results)


Arm S train=4431 val=492

===== TRAIN armS =====
  train=4431 val=492


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


Step,Training Loss
25,0.709736
50,0.421073
75,0.419607
100,0.473001
125,0.306633
150,0.345261
175,0.427349
200,0.290390
225,0.323588
250,0.441574


  [checkpoint] adapter_armS -> Drive
ARM S (filled schema in input)
    generating 1000 (batched, mode=json)...
   {'condition': 'S_json', 'n': 1000, 'per_q_f1ext': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.6666666666666666, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.6666666666666666, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8,

## 9c. Meaning 2 — paragraph vs structured passage (V7 fair, untuned)

Leak-proof content-vs-form test: the raw passage rewritten into organized prose (same content, reorganized, answer not pre-located; from `restructure_passages.py`). Untuned model only, compared head-to-head with the raw passage (C2). If structured < C2, imposing structure hurts even with full content; if ≈, it is neutral. Requires `restructured_passages.json` in the Drive input folder.

In [32]:
if 'untuned_structured_passage' in results:
    print('already in results, skipping (delete key to redo).')
elif os.path.exists(os.path.join(DATA, 'restructured_passages.json')):
    base = fresh_base_model()
    print('UNTUNED structured-passage (Meaning 2)')
    results['untuned_structured_passage'] = eval_structured_passage(base, 'restructured_passages.json')
    print('  ', results['untuned_structured_passage'])
    print('  vs raw passage C2:', results.get('untuned_C2', {}).get('f1_ext'))
    free(base)
    checkpoint_results(results)
else:
    print('(skipping Meaning 2: restructured_passages.json not in DATA. Run restructure_passages.py and upload it.)')

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

UNTUNED structured-passage (Meaning 2)
    generating 1000 (batched, structured-passage)...
   {'condition': 'structured_passage', 'n': 1000, 'per_q_f1ext': [0.75, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.0, 0.6666666666666666, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6666666666666666, 0.5, 0.6666666666666666, 0.0, 1.0, 1.0, 0.0, 0.9090909090909091, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.4, 0.0, 1.0, 0.0, 0.5, 0.0, 1.0, 0.3636363636363636, 0.0, 0.0, 0.0, 0.0, 0.5, 0.0, 0.8888888888888888, 1.0, 0.0, 0.5454545454545454, 0.0, 0.0, 0.0, 0.0, 0.2857142857142857, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6666666666666666, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5714285714285715, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.2857142857142857, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.4, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6666666666666666, 0.0, 1.0, 0.0, 0.0, 0

## 10. Summary and comparisons

In [33]:
import pandas as pd

ollama = {'C1': 0.369, 'C2': 0.906, 'C3': 0.343, 'C5': 0.360}  # untuned 3B, main study, F1_ext

def g(key, metric='f1_ext'):
    return results.get(key, {}).get(metric)

rows = []
for arm in ('untuned', 'arm1', 'arm2', 'arm4'):
    rows.append({
        'model': arm,
        'C1_f1ext': g(f'{arm}_C1'),
        'C3_f1ext': g(f'{arm}_C3'),
        'C5_f1ext': g(f'{arm}_C5'),
        'prose_f1ext': g(f'{arm}_prose'),
        'C2_f1ext': g(f'{arm}_C2'),
        'C3-C1': (None if g(f'{arm}_C3') is None or g(f'{arm}_C1') is None else round(g(f'{arm}_C3')-g(f'{arm}_C1'),4)),
        'C3-C5': (None if g(f'{arm}_C3') is None or g(f'{arm}_C5') is None else round(g(f'{arm}_C3')-g(f'{arm}_C5'),4)),
    })
df = pd.DataFrame(rows)
print('F1_ext by arm and condition (no passage):')
print(df.to_string(index=False))
print()
print('PARAGRAPH TARGET = untuned HF C2 (this run):', g('untuned_C2'), '  [Ollama main-study ref: 0.906]')
print('Arm1 C2 passage probe (did schema training break passage use?):', g('arm1_C2_probe'))
print()
print('READINGS')
print(' - Merit (Arm2 C3 - C5):', df[df.model=="arm2"]["C3-C5"].values)
print(' - Does structure beat no-structure after training (each arm C3 - C1):')
print(df[["model","C3-C1"]].to_string(index=False))
print(' - Paragraph parity: compare best C3 above against ~0.906. Large gap => structure is not a prose substitute.')
print()
print('DELIVERY FORM (empty content): alt-prose vs C3 schema, per arm:')
for arm in ('untuned','arm1','arm2','arm4'):
    pr, c3 = g(f'{arm}_prose'), g(f'{arm}_C3')
    if pr is not None and c3 is not None:
        print(f'  {arm}: prose={pr} schema_C3={c3} diff(prose-schema)={round(pr-c3,4)}')
print(' Form/content 2x2: [empty: alt-prose vs C3] [with-content: C2 passage vs Arm S filled]')

if 'armS_filled' in results:
    sS = results['armS_filled']['f1_ext']
    c2 = g('untuned_C2')
    print()
    print('Arm S (filled schema in input) F1_ext:', sS, '| untuned passage C2:', c2)
    print(' - Structure-vs-content: Arm S ~= C2 => content mattered, not form; Arm S << C2 => form matters beyond content.')
    print('   Caveat: filled schema is a lossy re-encoding of the passage, so content is not held perfectly equal.')

if 'untuned_structured_passage' in results:
    sp = results['untuned_structured_passage']['f1_ext']
    c2 = g('untuned_C2')
    print()
    print('Meaning 2 (V7 fair): structured-passage F1_ext:', sp, '| raw passage C2:', c2)
    d = None if (sp is None or c2 is None) else round(sp - c2, 4)
    print(' - structured - paragraph =', d, '| <0 => imposing structure hurts even with full content; ~0 => neutral.')

with open('multiarm_results.json','w') as f:
    json.dump(results, f, indent=2)
print('\nsaved multiarm_results.json (download it, plus adapter_arm1/2/4)')

F1_ext by arm and condition (no passage):
  model  C1_f1ext  C3_f1ext  C5_f1ext  prose_f1ext  C2_f1ext   C3-C1   C3-C5
untuned    0.1850    0.1276    0.1493       0.2528    0.3797 -0.0574 -0.0217
   arm1    0.2604    0.2549    0.2541       0.2532    0.3542 -0.0055  0.0008
   arm2    0.2649    0.2578    0.2662       0.2516    0.3620 -0.0071 -0.0084
   arm4    0.2642    0.2276    0.2178       0.2534    0.3876 -0.0366  0.0098

PARAGRAPH TARGET = untuned HF C2 (this run): 0.3797   [Ollama main-study ref: 0.906]
Arm1 C2 passage probe (did schema training break passage use?): None

READINGS
 - Merit (Arm2 C3 - C5): [-0.0084]
 - Does structure beat no-structure after training (each arm C3 - C1):
  model   C3-C1
untuned -0.0574
   arm1 -0.0055
   arm2 -0.0071
   arm4 -0.0366
 - Paragraph parity: compare best C3 above against ~0.906. Large gap => structure is not a prose substitute.

DELIVERY FORM (empty content): alt-prose vs C3 schema, per arm:
  untuned: prose=0.2528 schema_C3=0.1276 diff(pr

## 11. How to read the result

- **Merit (Arm 2, C3 − C5):** if matched clearly beats mismatched after training, the taxonomy label carries usable signal. This amplifies the main study's weak matched-over-random result (RQ2).
- **Rescue (any arm, C3 − C1, and vs untuned C3):** if a trained arm's matched-schema C3 rises above its own question-only C1 and above the untuned C3, training taught the model to use the structure. Supports the §6.6 delivery-failure account.
- **Paragraph parity:** compare the best trained C3 against the untuned passage C2 (~0.906). If C3 stays far below, structure — even trained — is not a substitute for prose without the content. If it approaches C2, the schema has real merit as an information carrier.
- **No narrowing (Arm 1 C2 probe):** if the Arm 1 model still scores near ~0.906 on the passage, schema-format training did not damage its core passage ability; a sharp drop indicates distribution narrowing (§2.3).
- **Floor (Arm 4):** how much of any C1 gain is just adapting the base model to terse QA, independent of structure.

Record the numbers into thesis §5 (new fine-tuning subsection) and update §6.6. Arm S (filled schema in the input) is the next experiment once the Claude population pipeline is built.